# System Tour: Data Ingestion & Backtesting

Interactive walkthrough of the two-layer data lake, trade simulator, and walk-forward backtest harness.

**Sections**
1. [Lake structure](#1-lake-structure)
2. [Tiingo equity data](#2-tiingo-equity-data)
3. [FRED macro data](#3-fred-macro-data)
4. [Walk-forward splits](#4-walk-forward-splits)
5. [Trade simulator](#5-trade-simulator)
6. [Performance metrics](#6-performance-metrics)
7. [Full walk-forward backtest](#7-full-walk-forward-backtest)
8. [Backtest report](#8-backtest-report)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, sys
from pathlib import Path

# Ensure the src tree is importable when running from notebooks/
ROOT = Path('..').resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.patches import Patch

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

print('Environment ready.')

In [ ]:
# Project imports
from quant.storage.lake import DataLake
from quant.storage.catalog import DataCatalog
from quant.backtest.walkforward import walkforward_splits
from quant.backtest.simulator import simulate
from quant.backtest.metrics import compute_metrics
from quant.backtest.harness import run_backtest
from quant.backtest.report import format_report, summary_table

lake = DataLake(ROOT / 'data')
catalog = DataCatalog(ROOT / 'data')
print('Lake root:', lake.root)

---
## 1 — Lake structure

The data lake has two layers:
- **raw/** — immutable per-day Parquet drops (never overwritten)
- **processed/** — hive-partitioned by `year=YYYY/month=MM`, rebuilt on each ingest run

In [ ]:
def lake_stats(root: Path) -> pd.DataFrame:
    rows = []
    for layer in ['raw', 'processed']:
        layer_path = root / layer
        if not layer_path.exists():
            continue
        for dataset in sorted(layer_path.iterdir()):
            if not dataset.is_dir():
                continue
            files = list(dataset.rglob('*.parquet'))
            size_mb = sum(f.stat().st_size for f in files) / 1_048_576
            rows.append({'layer': layer, 'dataset': dataset.name,
                         'files': len(files), 'size_mb': round(size_mb, 2)})
    return pd.DataFrame(rows)

stats = lake_stats(lake.root)
print(stats.to_string(index=False))

In [ ]:
if not stats.empty:
    fig, ax = plt.subplots(figsize=(8, 3))
    colors = {'raw': '#4C72B0', 'processed': '#DD8452'}
    for i, row in stats.iterrows():
        ax.barh(f"{row['layer']}/{row['dataset']}", row['size_mb'],
                color=colors.get(row['layer'], 'gray'))
    ax.set_xlabel('Size (MB)')
    ax.set_title('Data lake — file sizes by dataset')
    patches = [Patch(color=c, label=l) for l, c in colors.items()]
    ax.legend(handles=patches, loc='lower right')
    plt.tight_layout()
    plt.show()

---
## 2 — Tiingo equity data

Queried via DuckDB directly from the hive-partitioned Parquet files.

In [ ]:
try:
    eq = catalog.query(f"""
        SELECT symbol, timestamp, open, high, low, close, volume,
               adjClose, adjOpen, adjVolume, divCash, splitFactor
        FROM {catalog.table('equity_tiingo')}
        ORDER BY symbol, timestamp
    """)
    eq['timestamp'] = pd.to_datetime(eq['timestamp'])
    eq = eq.set_index('timestamp')
    print(f"{len(eq):,} rows | {eq.index.min().date()} → {eq.index.max().date()}")
    print(f"Symbols: {sorted(eq['symbol'].unique())}")
    eq.head(3)
except Exception as e:
    print(f'No Tiingo data yet: {e}')
    eq = pd.DataFrame()

In [ ]:
if not eq.empty:
    symbols = sorted(eq['symbol'].unique())
    # Coverage heatmap: month vs symbol
    eq_m = eq.copy()
    eq_m['ym'] = eq_m.index.to_period('M')
    coverage = eq_m.groupby(['symbol', 'ym']).size().unstack('symbol').notnull().astype(int)

    fig, ax = plt.subplots(figsize=(max(6, len(symbols) * 1.2), 5))
    im = ax.imshow(coverage.T, aspect='auto', cmap='Blues', vmin=0, vmax=1)
    ax.set_yticks(range(len(symbols)))
    ax.set_yticklabels(symbols)
    step = max(1, len(coverage) // 12)
    ax.set_xticks(range(0, len(coverage), step))
    ax.set_xticklabels([str(p) for p in coverage.index[::step]], rotation=45, ha='right')
    ax.set_title('Tiingo coverage heatmap (blue = data present)')
    plt.tight_layout()
    plt.show()

In [ ]:
if not eq.empty:
    sym = symbols[0]
    s = eq[eq['symbol'] == sym].copy()

    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

    # Adjusted close
    axes[0].plot(s.index, s['adjClose'], linewidth=1)
    axes[0].set_ylabel('Adj Close ($)')
    axes[0].set_title(f'{sym} — price, volume, corporate actions')

    # Volume
    axes[1].bar(s.index, s['adjVolume'], width=1, alpha=0.6)
    axes[1].set_ylabel('Adj Volume')
    axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

    # Dividends and splits
    ax3 = axes[2]
    divs = s[s['divCash'] > 0]
    splits = s[s['splitFactor'] != 1.0]
    ax3.bar(divs.index, divs['divCash'], width=5, color='steelblue', label='divCash')
    ax3.set_ylabel('divCash ($)', color='steelblue')
    ax3_r = ax3.twinx()
    ax3_r.scatter(splits.index, splits['splitFactor'], color='darkorange', zorder=5, label='splitFactor')
    ax3_r.set_ylabel('splitFactor', color='darkorange')
    ax3.set_xlabel('Date')
    lines1, labels1 = ax3.get_legend_handles_labels()
    lines2, labels2 = ax3_r.get_legend_handles_labels()
    ax3.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

    plt.tight_layout()
    plt.show()

---
## 3 — FRED macro data

In [ ]:
try:
    fred = catalog.query(f"""
        SELECT series_id, timestamp, value
        FROM {catalog.table('macro_fred')}
        ORDER BY series_id, timestamp
    """)
    fred['timestamp'] = pd.to_datetime(fred['timestamp'])
    series_list = sorted(fred['series_id'].unique())
    print(f"{len(fred):,} rows | {len(series_list)} series")
    print('Series:', series_list)
    fred.head(3)
except Exception as e:
    print(f'No FRED data yet: {e}')
    fred = pd.DataFrame()
    series_list = []

In [ ]:
if not fred.empty:
    n = len(series_list)
    cols = min(2, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows), squeeze=False)
    for idx, sid in enumerate(series_list):
        ax = axes[idx // cols][idx % cols]
        s = fred[fred['series_id'] == sid].set_index('timestamp')['value']
        ax.plot(s.index, s.values, linewidth=1)
        ax.set_title(sid)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    # Hide unused subplots
    for idx in range(n, rows * cols):
        axes[idx // cols][idx % cols].set_visible(False)
    fig.suptitle('FRED macro series', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

---
## 4 — Walk-forward splits

The `walkforward_splits` generator implements **purged** rolling CV:
- **Purge**: remove training samples whose label window overlaps the test period  
- **Embargo**: additional buffer after purging to defeat residual serial correlation

In [ ]:
N, TRAIN, TEST, STEP, HORIZON, EMBARGO = 200, 100, 20, 20, 5, 3

splits = list(walkforward_splits(
    n_samples=N,
    train_window=TRAIN,
    test_window=TEST,
    step=STEP,
    label_horizon=HORIZON,
    embargo=EMBARGO,
))
print(f"{len(splits)} folds | train_window={TRAIN}, test_window={TEST}, "
      f"step={STEP}, label_horizon={HORIZON}, embargo={EMBARGO}")
for i, (tr, te) in enumerate(splits[:3]):
    print(f"  fold {i}: train [{tr[0]}..{tr[-1]}] ({len(tr)} samples) | "
          f"test [{te[0]}..{te[-1]}] ({len(te)} samples)")
if len(splits) > 3:
    print('  ...')

In [ ]:
fig, ax = plt.subplots(figsize=(12, max(3, len(splits) * 0.35)))

for fold_idx, (tr, te) in enumerate(splits):
    y = fold_idx
    # Full nominal train window (grey)
    nominal_start = te[0] - TRAIN
    ax.barh(y, TRAIN, left=nominal_start, height=0.6, color='#cccccc', alpha=0.5)
    # Actual purged + embargoed train (blue)
    if len(tr) > 0:
        ax.barh(y, len(tr), left=tr[0], height=0.6, color='#4C72B0', alpha=0.8)
    # Gap (purge + embargo) shown in red
    purge_start = te[0] - HORIZON - EMBARGO
    ax.barh(y, HORIZON + EMBARGO, left=purge_start, height=0.6, color='#C44E52', alpha=0.6)
    # Test window (green)
    ax.barh(y, len(te), left=te[0], height=0.6, color='#55A868', alpha=0.8)

ax.set_yticks(range(len(splits)))
ax.set_yticklabels([f'Fold {i}' for i in range(len(splits))])
ax.set_xlabel('Sample index')
ax.set_title('Purged walk-forward splits')
legend_patches = [
    Patch(color='#cccccc', alpha=0.5, label='Nominal train window'),
    Patch(color='#4C72B0', alpha=0.8, label='Effective train (post purge+embargo)'),
    Patch(color='#C44E52', alpha=0.6, label='Purge + embargo gap'),
    Patch(color='#55A868', alpha=0.8, label='Test window'),
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

---
## 5 — Trade simulator

A simple MA-crossover signal on real Tiingo data — just to see the simulator in action.

**Signal**: +1 when close > MA(20), -1 otherwise.  
**Execution**: fills at next-bar open with slippage + commission.

In [ ]:
if not eq.empty:
    sym = symbols[0]
    prices = eq[eq['symbol'] == sym][['open', 'high', 'low', 'close', 'volume']].copy()
    prices = prices.sort_index().dropna()

    ma20 = prices['close'].rolling(20).mean()
    raw_sig = np.where(prices['close'] > ma20, 1, -1)
    signals = pd.Series(raw_sig, index=prices.index, dtype=int)
    signals.iloc[:20] = 0   # flat during MA warmup

    equity, trade_log = simulate(
        prices=prices,
        signals=signals,
        initial_capital=100_000,
        commission_per_share=0.005,
        slippage_bps=5.0,
        liquidity_cap=0.10,
    )
    print(f'Trades: {len(trade_log)} | Final equity: ${equity.iloc[-1]:,.0f}')
    trade_log.tail(5)

In [ ]:
if not eq.empty:
    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

    # Price + MA
    axes[0].plot(prices.index, prices['close'], linewidth=1, label='Close')
    axes[0].plot(prices.index, ma20, linewidth=1, linestyle='--', label='MA(20)')
    # Entry/exit markers
    if not trade_log.empty:
        dates = trade_log['date']
        axes[0].scatter(dates, trade_log['entry_price'], marker='^', color='green',
                        s=40, zorder=5, label='Entry')
        axes[0].scatter(dates, trade_log['exit_price'], marker='v', color='red',
                        s=40, zorder=5, label='Exit')
    axes[0].set_ylabel('Price ($)')
    axes[0].set_title(f'{sym} — MA(20) crossover strategy')
    axes[0].legend(fontsize=9)

    # Equity curve
    axes[1].plot(equity.index, equity.values / 1000, linewidth=1, color='#4C72B0')
    axes[1].axhline(100, color='black', linewidth=0.5, linestyle='--')
    axes[1].set_ylabel('Portfolio ($k)')

    # Drawdown
    peak = equity.cummax()
    dd = (equity - peak) / peak * 100
    axes[2].fill_between(dd.index, dd.values, 0, color='#C44E52', alpha=0.4)
    axes[2].set_ylabel('Drawdown (%)')
    axes[2].set_xlabel('Date')

    plt.tight_layout()
    plt.show()

---
## 6 — Performance metrics

In [ ]:
if not eq.empty:
    returns = equity.pct_change().dropna()
    m = compute_metrics(returns, trade_log=trade_log if not trade_log.empty else None)

    print('=== MA(20) strategy metrics ===')
    for k, v in m.items():
        if k in ('total_return', 'annualized_return', 'max_drawdown', 'hit_rate'):
            print(f'  {k:<20} {v:.2%}')
        elif k == 'profit_factor':
            print(f'  {k:<20} {v:.3f}' if not (isinstance(v, float) and (v != v or v == float("inf"))) else f'  {k:<20} {v}')
        else:
            print(f'  {k:<20} {v:.3f}')

In [ ]:
if not eq.empty:
    # Monthly returns heatmap
    monthly = returns.resample('ME').apply(lambda r: (1 + r).prod() - 1)
    monthly_df = monthly.to_frame('ret')
    monthly_df['year'] = monthly_df.index.year
    monthly_df['month'] = monthly_df.index.month
    pivot = monthly_df.pivot(index='year', columns='month', values='ret')
    pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                     'Jul','Aug','Sep','Oct','Nov','Dec']

    import matplotlib.colors as mcolors
    cmap = mcolors.TwoSlopeNorm(vcenter=0,
                                 vmin=pivot.values[~np.isnan(pivot.values)].min(),
                                 vmax=pivot.values[~np.isnan(pivot.values)].max())

    fig, ax = plt.subplots(figsize=(14, max(3, len(pivot) * 0.5)))
    im = ax.imshow(pivot.values, cmap='RdYlGn', norm=cmap, aspect='auto')
    ax.set_xticks(range(12))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot)))
    ax.set_yticklabels(pivot.index)
    for r in range(len(pivot)):
        for c in range(12):
            v = pivot.values[r, c]
            if not np.isnan(v):
                ax.text(c, r, f'{v:.1%}', ha='center', va='center', fontsize=8)
    plt.colorbar(im, ax=ax, format=mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.set_title(f'{sym} MA(20) — monthly returns heatmap')
    plt.tight_layout()
    plt.show()

---
## 7 — Full walk-forward backtest

Uses `run_backtest` from the harness with a `LogisticRegression` model and lagged return features.

In [ ]:
if not eq.empty:
    from sklearn.linear_model import LogisticRegression

    sym = symbols[0]
    prices_bt = eq[eq['symbol'] == sym][['open', 'high', 'low', 'close', 'volume']].copy()
    prices_bt = prices_bt.sort_index().dropna()

    # Features: lagged log-returns (1, 2, 3, 5, 10 bars)
    lags = [1, 2, 3, 5, 10]
    log_ret = np.log(prices_bt['close'] / prices_bt['close'].shift(1))
    features = pd.DataFrame(
        {f'lag_{l}': log_ret.shift(l) for l in lags},
        index=prices_bt.index,
    ).dropna()
    prices_bt = prices_bt.loc[features.index]

    # Label: sign of 2-bar forward return (signal at t fills t+1, earns t+1→t+2)
    fwd2 = np.log(prices_bt['close'].shift(-2) / prices_bt['close'].shift(-1))
    labels = np.sign(fwd2).shift(0)  # aligned to features index

    # Drop rows where we can't form a label
    valid = features.index.intersection(labels.dropna().index)
    features = features.loc[valid]
    prices_bt = prices_bt.loc[valid]

    model = LogisticRegression(C=1.0, max_iter=500, random_state=42)

    print(f'Running backtest on {len(features)} bars of {sym}...')
    result = run_backtest(
        model=model,
        features=features,
        prices=prices_bt,
        train_window=252,
        test_window=63,
        step=63,
        label_horizon=2,
        embargo=5,
        initial_capital=100_000,
        commission_per_share=0.005,
        slippage_bps=5.0,
    )
    print(f'Done — {len(result.fold_metrics)} folds, {len(result.trade_log)} trades')

In [ ]:
if not eq.empty and 'result' in dir():
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

    # OOS equity curve
    ec = result.equity_curve
    axes[0].plot(ec.index, ec.values / 1000, linewidth=1, color='#4C72B0', label='OOS equity')
    axes[0].axhline(100, color='black', linewidth=0.5, linestyle='--')

    # Shade each fold
    if result.fold_metrics:
        fm = result.fold_metrics
        fold_dates = []
        for i, fold in enumerate(fm):
            if 'test_start' in fold and 'test_end' in fold:
                fold_dates.append((fold['test_start'], fold['test_end'], i))
        for start, end, idx in fold_dates:
            color = '#4C72B0' if idx % 2 == 0 else '#DD8452'
            axes[0].axvspan(start, end, alpha=0.05, color=color)

    axes[0].set_ylabel('Portfolio ($k)')
    axes[0].set_title(f'{sym} — LogisticRegression walk-forward backtest (OOS)')
    axes[0].legend()

    # Drawdown
    peak = ec.cummax()
    dd = (ec - peak) / peak * 100
    axes[1].fill_between(dd.index, dd.values, 0, color='#C44E52', alpha=0.4)
    axes[1].set_ylabel('Drawdown (%)')
    axes[1].set_xlabel('Date')

    plt.tight_layout()
    plt.show()

In [ ]:
if not eq.empty and 'result' in dir():
    # Per-fold OOS Sharpe
    fold_sharpes = [f.get('sharpe', float('nan')) for f in result.fold_metrics]
    fig, ax = plt.subplots(figsize=(max(6, len(fold_sharpes) * 0.6), 3))
    colors = ['#55A868' if s > 0 else '#C44E52' for s in fold_sharpes]
    ax.bar(range(len(fold_sharpes)), fold_sharpes, color=colors)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Fold')
    ax.set_ylabel('Sharpe')
    ax.set_title('OOS Sharpe by fold')
    plt.tight_layout()
    plt.show()

---
## 8 — Backtest report

In [ ]:
if not eq.empty and 'result' in dir():
    print(format_report(result))

In [ ]:
if not eq.empty and 'result' in dir():
    tbl = summary_table(result)
    display(tbl.style
               .format({
                   'OOS': lambda v: f'{v:.2%}' if abs(v) < 5 else f'{v:.3f}',
                   'IS':  lambda v: f'{v:.2%}' if abs(v) < 5 else f'{v:.3f}',
               })
               .background_gradient(cmap='RdYlGn', axis=None))

In [ ]:
if not eq.empty and 'result' in dir():
    # IS/OOS gap — key overfit diagnostic
    oos = result.oos_metrics
    is_ = result.is_metrics
    keys = ['sharpe', 'total_return', 'hit_rate', 'max_drawdown']
    gaps = {k: is_.get(k, float('nan')) - oos.get(k, float('nan')) for k in keys}

    print('IS/OOS gap (positive = IS beats OOS, i.e. potential overfit):')
    for k, g in gaps.items():
        flag = '  <<< WARN' if abs(g) > 0.5 and k == 'sharpe' else ''
        print(f'  {k:<20} gap = {g:+.3f}{flag}')